# OTEL Logs — Silver Pipeline

Spark Declarative Pipeline that creates silver tables from raw OTEL log data.

| Table | Description |
| --- | --- |
| `logs_flattened` | All logs with timestamps converted, resource attrs promoted, severity parsed from body |
| `logs_coredns_parsed` | CoreDNS logs with regex-parsed DNS query fields |
| `logs_errors` | Error/warning/fatal logs filtered for monitoring |

In [ ]:
import dlt
from pyspark.sql import functions as F

In [ ]:
source_catalog = spark.conf.get("source_catalog")
source_schema = spark.conf.get("source_schema")
table_prefix = spark.conf.get("table_prefix")
source_table = f"{source_catalog}.{source_schema}.{table_prefix}_otel_logs"

In [ ]:
# Severity is parsed from the log body since the OTel severity_text field is
# unpopulated. This CASE expression covers 12+ log formats found in a typical
# Kubernetes cluster (CoreDNS, Java log4j, Go klog, .NET, Envoy, etc.).
SEVERITY_EXPR = (
    "CASE"
    "  WHEN body LIKE '[INFO]%'                   THEN 'INFO'"
    "  WHEN body LIKE '[WARNING]%'                THEN 'WARN'"
    "  WHEN body LIKE '[ERROR]%'                  THEN 'ERROR'"
    "  WHEN body LIKE '[FATAL]%'                  THEN 'FATAL'"
    "  WHEN body LIKE '[Error]%'                  THEN 'ERROR'"
    "  WHEN body LIKE '[Warning]%'                THEN 'WARN'"
    "  WHEN body LIKE '[Info]%'                   THEN 'INFO'"
    "  WHEN body RLIKE '^I\\\\d{4} '                THEN 'INFO'"
    "  WHEN body RLIKE '^W\\\\d{4} '                THEN 'WARN'"
    "  WHEN body RLIKE '^E\\\\d{4} '                THEN 'ERROR'"
    "  WHEN body RLIKE '^F\\\\d{4} '                THEN 'FATAL'"
    "  WHEN body RLIKE '^E, \\\\['                  THEN 'ERROR'"
    "  WHEN body RLIKE '^W, \\\\['                  THEN 'WARN'"
    "  WHEN body RLIKE '^I, \\\\['                  THEN 'INFO'"
    "  WHEN body LIKE 'info:%'                    THEN 'INFO'"
    "  WHEN body LIKE 'warn:%'                    THEN 'WARN'"
    "  WHEN body LIKE 'fail:%'                    THEN 'ERROR'"
    "  WHEN body LIKE 'crit:%'                    THEN 'FATAL'"
    "  WHEN body RLIKE '\\\\]\\\\[info\\\\]'            THEN 'INFO'"
    "  WHEN body RLIKE '\\\\]\\\\[warning\\\\]'         THEN 'WARN'"
    "  WHEN body RLIKE '\\\\]\\\\[error\\\\]'           THEN 'ERROR'"
    "  WHEN body RLIKE '\\\\]\\\\[critical\\\\]'        THEN 'FATAL'"
    "  WHEN body LIKE '%\"level\":\"info\"%'          THEN 'INFO'"
    "  WHEN body LIKE '%\"level\":\"warn\"%'          THEN 'WARN'"
    "  WHEN body LIKE '%\"level\":\"warning\"%'       THEN 'WARN'"
    "  WHEN body LIKE '%\"level\":\"error\"%'         THEN 'ERROR'"
    "  WHEN body LIKE '\\tat %'                    THEN 'TRACE_CONTINUATION'"
    "  WHEN body LIKE 'Caused by:%'               THEN 'ERROR'"
    "  WHEN body RLIKE '^\\\\d{2}:\\\\d{2}:\\\\d{2}\\\\.\\\\d+ \\\\[info\\\\]'    THEN 'INFO'"
    "  WHEN body RLIKE '^\\\\d{2}:\\\\d{2}:\\\\d{2}\\\\.\\\\d+ \\\\[error\\\\]'   THEN 'ERROR'"
    "  WHEN body RLIKE '^\\\\d{2}:\\\\d{2}:\\\\d{2}\\\\.\\\\d+ \\\\[warning\\\\]' THEN 'WARN'"
    "  WHEN body RLIKE '^\\\\[\\\\d{4}-\\\\d{2}-\\\\d{2} [^\\\\]]+\\\\] INFO'  THEN 'INFO'"
    "  WHEN body RLIKE '^\\\\[\\\\d{4}-\\\\d{2}-\\\\d{2} [^\\\\]]+\\\\] WARN'  THEN 'WARN'"
    "  WHEN body RLIKE '^\\\\[\\\\d{4}-\\\\d{2}-\\\\d{2} [^\\\\]]+\\\\] ERROR' THEN 'ERROR'"
    "  WHEN body LIKE 'Error:%'                   THEN 'ERROR'"
    "  WHEN body LIKE 'Error %'                   THEN 'ERROR'"
    "  ELSE 'UNCLASSIFIED'"
    " END AS parsed_severity"
)

In [ ]:
@dlt.table(
    comment="All OTel logs with flattened attributes and severity parsed from body text",
)
def logs_flattened():
    return (
        spark.readStream.table(source_table)
        .filter(F.col("time_unix_nano").isNotNull())
        .selectExpr(
            "TIMESTAMP_MICROS(CAST(time_unix_nano / 1000 AS BIGINT)) AS log_timestamp",
            "TIMESTAMP_MICROS(CAST(observed_time_unix_nano / 1000 AS BIGINT)) AS observed_timestamp",
            "ROUND((observed_time_unix_nano - time_unix_nano) / 1e6, 2) AS ingestion_lag_ms",
            SEVERITY_EXPR,
            "body",
            "LENGTH(body) AS body_length",
            "CASE WHEN body LIKE '\\tat %' OR body LIKE 'Caused by:%' THEN TRUE ELSE FALSE END AS is_stack_trace_line",
            # K8s resource hierarchy
            "resource.attributes['k8s.node.name'] AS node",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.deployment.name'] AS deployment",
            "resource.attributes['k8s.daemonset.name'] AS daemonset",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "resource.attributes['k8s.container.name'] AS container",
            "CAST(resource.attributes['k8s.container.restart_count'] AS INT) AS container_restart_count",
            "resource.attributes['k8s.replicaset.name'] AS replicaset",
            "resource.attributes['k8s.cluster.uid'] AS cluster_uid",
            # Service identity
            "resource.attributes['service.name'] AS service",
            "resource.attributes['service.namespace'] AS service_namespace",
            "resource.attributes['service.instance.id'] AS service_instance_id",
            "resource.attributes['service.version'] AS service_version",
            "resource.attributes['container.image.name'] AS image_name",
            "resource.attributes['container.image.tag'] AS image_tag",
            # Log-level attributes
            "attributes['log.file.path'] AS log_file_path",
            "attributes['log.iostream'] AS log_iostream",
            "attributes['logtag'] AS logtag",
            # Trace correlation
            "NULLIF(trace_id, '') AS trace_id",
            "NULLIF(span_id, '') AS span_id",
        )
    )

In [ ]:
@dlt.table(
    comment="CoreDNS logs with parsed DNS query details",
)
def logs_coredns_parsed():
    return (
        dlt.read_stream("logs_flattened")
        .filter(
            (F.col("service") == "coredns") & (F.col("body").like("[INFO]%"))
        )
        .selectExpr(
            "log_timestamp",
            "observed_timestamp",
            "ingestion_lag_ms",
            "node",
            "namespace",
            "pod",
            "REGEXP_EXTRACT(body, '\\\\[INFO\\\\] ([^:]+):', 1) AS client_ip",
            "TRY_CAST(REGEXP_EXTRACT(body, '\\\\[INFO\\\\] [^:]+:(\\\\d+)', 1) AS INT) AS client_port",
            "TRY_CAST(REGEXP_EXTRACT(body, '- (\\\\d+) \"', 1) AS INT) AS query_id",
            "REGEXP_EXTRACT(body, '\"(\\\\w+) IN ', 1) AS query_type",
            "REGEXP_EXTRACT(body, ' IN ([^ ]+) ', 1) AS domain",
            "REGEXP_EXTRACT(body, ' (udp|tcp) ', 1) AS protocol",
            "TRY_CAST(REGEXP_EXTRACT(body, ' (?:udp|tcp) (\\\\d+) ', 1) AS INT) AS query_size",
            "REGEXP_EXTRACT(body, '\" (\\\\w+) ', 1) AS response_code",
            "REGEXP_EXTRACT(body, '\" \\\\w+ ([^ ]+) ', 1) AS response_flags",
            "TRY_CAST(REGEXP_EXTRACT(body, '\" \\\\w+ [^ ]+ (\\\\d+) ', 1) AS INT) AS response_size",
            "TRY_CAST(REGEXP_EXTRACT(body, '([\\\\d.]+)s$', 1) AS DOUBLE) AS duration_seconds",
            "body",
        )
    )

In [ ]:
@dlt.table(
    comment="Error and warning logs for monitoring and alerting",
)
def logs_errors():
    return (
        dlt.read_stream("logs_flattened")
        .filter(F.col("parsed_severity").isin("ERROR", "WARN", "FATAL"))
        .select(
            "log_timestamp",
            "observed_timestamp",
            "ingestion_lag_ms",
            "parsed_severity",
            "node",
            "namespace",
            "deployment",
            "daemonset",
            "pod",
            "container",
            "container_restart_count",
            "service",
            "service_namespace",
            "image_name",
            "image_tag",
            "body",
            "body_length",
            "is_stack_trace_line",
            "log_file_path",
            "log_iostream",
            "trace_id",
            "span_id",
        )
    )